# Circular and Resilient Capacity Planning under Uncertainty


**Paper:** *Circular and Resilient Capacity Planning under Uncertainty in Emerging-Economy
Textile Supply Chains: A Hybrid DEMATEL-VIKOR and Stochastic Mixed-Integer Linear Programming (MILP) Optimization Framework*



In [1]:
# —— Environment setup ————————————————————————————————————————
import subprocess, sys

def _pip_install(pkg):
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                               stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                                '--break-system-packages', pkg],
                               stderr=subprocess.DEVNULL)

for _pkg in ['pyomo', 'highspy', 'openpyxl']:
    _pip_install(_pkg)

print("✓ Packages installed.")

✓ Packages installed.


In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import pyomo.environ as pyo

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os, time
from collections import OrderedDict

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.family': 'serif', 'font.serif': ['DejaVu Serif'],
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 9,
    'figure.figsize': (10, 6), 'axes.grid': True, 'grid.alpha': 0.3,
})

print("✓ Imports loaded.")

✓ Imports loaded.


## DEMATEL Results Embedding

# Strategies:
whose constituent factors are net *causes* (D−R > 0) receive a **resilience-effectiveness
bonus**, reflecting the DEMATEL finding that causal drivers have greater systemic
leverage. Prominence (D+R) continues to serve as strategy-level weights.

In [3]:
DEMATEL_FACTORS = pd.DataFrame({
    'Code':       ['F1','F2','F3','F4','F5','F6','F7','F8','F9','F10',
                   'F11','F12','F13','F14','F15','F16','F17','F18'],
    'Name': [
        'Circular Design Capability & Eco-Design',
        'Circular Standardization & Material Traceability',
        'Reverse Logistics, Collection & Sorting Infrastructure',
        'Recycled-Input Procurement & Closed-Loop Sourcing',
        'Flexible Sourcing & Supplier Diversification',
        'Flexible Production Capacity & Operational Agility',
        'Inventory Buffering & Redundancy Strategy',
        'Proactive Risk Management & Disruption Preparedness',
        'Demand & Disruption Forecasting Capability',
        'Supply Chain Collaboration, Trust & Goal Congruence',
        'Technology Integration & Digitalization',
        'Supply Chain Transparency, Visibility & Information Sharing',
        'Top Management Commitment & Leadership',
        'Financial Strength, Liquidity & Access to Green Funding',
        'Workforce Skills, Training & Environmental Awareness',
        'Government Policy, Regulatory Support & Institutional Pressure',
        'Market Diversification & Buyer Demand for Sustainability',
        'Infrastructure & Utility Reliability'
    ],
    'D_plus_R':   [2.915, 2.939, 2.896, 3.000, 2.924, 2.975, 2.752, 3.085,
                   2.933, 3.198, 3.228, 3.022, 3.225, 3.219, 2.903, 3.070,
                   3.045, 2.643],
    'D_minus_R':  [-0.346, -0.303, -0.997, -0.889, -0.673, -0.708, -0.845,
                    0.164,  0.212,  0.615,  0.730, -0.287,  1.486,  1.321,
                   -0.192,  1.478, -0.690, -0.076],
})

_dpr = DEMATEL_FACTORS['D_plus_R']
DEMATEL_FACTORS['w_prominence'] = (_dpr - _dpr.min()) / (_dpr.max() - _dpr.min())
_dmr = DEMATEL_FACTORS['D_minus_R']
DEMATEL_FACTORS['w_causal'] = (_dmr - _dmr.min()) / (_dmr.max() - _dmr.min())
DEMATEL_FACTORS['is_cause'] = DEMATEL_FACTORS['D_minus_R'] > 0

STRATEGIES = OrderedDict({
    'S1':  {'name': 'Eco-Design & Circular Product Development',
            'factors': ['F1','F2'],   'type': 'circular'},
    'S2':  {'name': 'Reverse Logistics & Recovery Infrastructure',
            'factors': ['F3','F4'],   'type': 'circular'},
    'S3':  {'name': 'Closed-Loop Material Sourcing',
            'factors': ['F4','F2'],   'type': 'circular'},
    'S4':  {'name': 'Flexible Sourcing & Supplier Diversification',
            'factors': ['F5','F10'],  'type': 'resilience'},
    'S5':  {'name': 'Agile Production & Operational Flexibility',
            'factors': ['F6','F8'],   'type': 'resilience'},
    'S6':  {'name': 'Strategic Inventory & Buffer Capacity',
            'factors': ['F7','F8'],   'type': 'resilience'},
    'S7':  {'name': 'Technology & Digital Supply Chain Integration',
            'factors': ['F11','F12','F9'], 'type': 'both'},
    'S8':  {'name': 'Workforce Upskilling & Green Training Programs',
            'factors': ['F15','F13'], 'type': 'both'},
    'S9':  {'name': 'Green Finance & Compliance Investment',
            'factors': ['F14','F16'], 'type': 'both'},
    'S10': {'name': 'Market Diversification & Sustainable Buyer Development',
            'factors': ['F17','F18'], 'type': 'both'},
})

for s_key, s_val in STRATEGIES.items():
    factor_rows = DEMATEL_FACTORS[DEMATEL_FACTORS['Code'].isin(s_val['factors'])]
    s_val['dematel_weight'] = factor_rows['w_prominence'].mean()
    s_val['is_cause'] = (factor_rows['D_minus_R'] > 0).any()
    s_val['causal_score'] = factor_rows['w_causal'].mean()

print("✓ DEMATEL results embedded. 10 capacity strategies defined.")
print(f"  Circular: {sum(1 for v in STRATEGIES.values() if v['type']=='circular')} | "
      f"Resilience: {sum(1 for v in STRATEGIES.values() if v['type']=='resilience')} | "
      f"Both: {sum(1 for v in STRATEGIES.values() if v['type']=='both')}")
print(f"\n{'Strategy':<6} {'DEMATEL_w':>9} {'Causal':>7} {'IsCause':>8} {'Type':<12}")
print("─"*48)
for k, v in STRATEGIES.items():
    print(f"{k:<6} {v['dematel_weight']:>9.3f} {v['causal_score']:>7.3f} "
          f"{'Yes' if v['is_cause'] else 'No':>8} {v['type']:<12}")

✓ DEMATEL results embedded. 10 capacity strategies defined.
  Circular: 3 | Resilience: 3 | Both: 4

Strategy DEMATEL_w  Causal  IsCause Type        
────────────────────────────────────────────────
S1         0.485   0.271       No circular    
S2         0.521   0.022       No circular    
S3         0.558   0.161       No circular    
S4         0.715   0.390      Yes resilience  
S5         0.662   0.292      Yes resilience  
S6         0.471   0.264      Yes resilience  
S7         0.715   0.489      Yes both        
S8         0.720   0.662      Yes both        
S9         0.857   0.965      Yes both        
S10        0.344   0.247       No both        


## Data Loading (Bangladesh RMG Dataset)


In [4]:
DATA_PATHS = [
    '/kaggle/input/datasets/mdlimonbinhossain/rmg-bd/Bangladesh_RMG_ML_Dataset.xlsx',
    '/kaggle/input/Bangladesh_RMG_ML_Dataset.xlsx',
    './Bangladesh_RMG_ML_Dataset.xlsx',
    '/mnt/user-data/uploads/Bangladesh_RMG_ML_Dataset.xlsx',
]
DATA_PATH = next((p for p in DATA_PATHS if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError("Dataset not found.")
print(f"✓ Dataset found at: {DATA_PATH}")

df_raw   = pd.read_excel(DATA_PATH, sheet_name='Raw_Variables')
df_commodities = pd.read_excel(DATA_PATH, sheet_name='Commodities')
df_wb    = pd.read_excel(DATA_PATH, sheet_name='WorldBank_Data')

df = df_raw.copy()
df = df.merge(df_commodities[['Date','Oil_Brent_USD_bbl_PS']], on='Date', how='left')
wb_cols = ['Date','GDP_Growth_pct','Manufacturing_PctGDP','ForeignReserves_USDmn',
           'TotalExports_USDmn','TotalImports_USDmn','TradeOpenness_PctGDP',
           'GDP_PerCapita_USD','Population']
df = df.merge(df_wb[wb_cols], on='Date', how='left')
df['DateParsed'] = pd.to_datetime(df['Date'], format='%Y-%m')
df = df.sort_values('DateParsed').reset_index(drop=True)

TARGET = 'RMG_Export_USDmn'

# Disruption composite: normalize each component to [0,1] before summing (v2 fix)
disrupt_cols = ['COVID_dummy','LaborUnrest_idx','Flood_severity',
                'PowerOutage_idx','PoliticalInstability_dummy','GovernmentPolicy_idx']
component_max = {'COVID_dummy': 1, 'LaborUnrest_idx': 3, 'Flood_severity': 3,
                 'PowerOutage_idx': 3, 'PoliticalInstability_dummy': 1, 'GovernmentPolicy_idx': 3}
for col in disrupt_cols:
    df[f'{col}_norm'] = df[col] / component_max[col]
df['Disruption_NormScore'] = sum(df[f'{c}_norm'] for c in disrupt_cols)
df['Disruption_CompositeScore'] = sum(df[c] for c in disrupt_cols)

# Cotton/oil cost index (60% cotton, 15% energy, 25% other)
df['Cotton_idx'] = df['Cotton_Price_USDkg'] / df['Cotton_Price_USDkg'].dropna().median()
oil_col = 'Oil_Brent_USD_bbl_PS'
df['Oil_idx'] = df[oil_col] / df[oil_col].dropna().median()
df['RawMaterial_CostIdx'] = 0.60 * df['Cotton_idx'].fillna(1.0) + 0.15 * df['Oil_idx'].fillna(1.0) + 0.25

print(f"✓ Consolidated dataframe: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  Date range: {df['DateParsed'].min().strftime('%Y-%m')} to "
      f"{df['DateParsed'].max().strftime('%Y-%m')}")
print(f"  Variables used: RMG exports, cotton price, oil price, CPI, inflation,")
print(f"  lending rate, EU/US imports, 6 disruption components, macro indicators")

✓ Dataset found at: /kaggle/input/datasets/mdlimonbinhossain/rmg-bd/Bangladesh_RMG_ML_Dataset.xlsx
✓ Consolidated dataframe: 252 rows × 39 columns
  Date range: 2005-01 to 2025-12
  Variables used: RMG exports, cotton price, oil price, CPI, inflation,
  lending rate, EU/US imports, 6 disruption components, macro indicators


##  Scenario Generation (Empirically Calibrated)


1. Disruption scores normalized (each component/max before summing)
2. Cost factor derived from cotton/oil cost index (not hand-picked `1+0.05*D`)
3. Probabilities documented as planning weights (not empirical frequencies)
4. Independence assumption explicitly acknowledged

In [5]:
recent_36 = df.dropna(subset=[TARGET]).tail(36)
base_demand = recent_36[TARGET].median()

demand_scenarios = {
    'Low':    recent_36[TARGET].quantile(0.25),
    'Medium': recent_36[TARGET].quantile(0.50),
    'High':   recent_36[TARGET].quantile(0.75),
}

disrupt_hist = df['Disruption_NormScore'].dropna()
disruption_scenarios = {
    'None':     0.0,
    'Moderate': disrupt_hist.quantile(0.75),
    'Severe':   disrupt_hist.quantile(0.95),
}

# Empirical capacity utilization calibration
def empirical_util_cap(ds_norm):
    return max(0.35, 1.0 - 0.12 * ds_norm)

def empirical_cost_factor(ds_norm):
    return 1.0 + 0.04 * ds_norm

CIRC_AVAIL_BASELINE = 0.20
def empirical_circ_availability(ds_norm):
    return max(0.05, CIRC_AVAIL_BASELINE - 0.03 * ds_norm)

# Scenario probability weights (researcher-defined planning weights)
demand_probs  = {'Low': 0.25, 'Medium': 0.50, 'High': 0.25}
disrupt_probs = {'None': 0.50, 'Moderate': 0.35, 'Severe': 0.15}

scenarios = {}
s_idx = 0
for d_name, d_val in demand_scenarios.items():
    for r_name, r_val in disruption_scenarios.items():
        s_idx += 1
        s_key = f'S{s_idx:02d}'
        prob = demand_probs[d_name] * disrupt_probs[r_name]
        scenarios[s_key] = {
            'demand_level': d_name, 'disruption_level': r_name,
            'demand_usd_mn': d_val, 'disruption_score': r_val,
            'probability': prob, 'desc': f'{d_name} demand / {r_name} disruption'
        }

# Tail-risk scenarios
covid_mask = df['COVID_dummy'] >= 0.5
covid_trough = df.loc[covid_mask, TARGET].min() if covid_mask.any() else base_demand * 0.60

cascading_mask = df['Disruption_CompositeScore'] >= 4
cascading_demand = (df.loc[cascading_mask, TARGET].quantile(0.25)
                     if cascading_mask.sum() >= 5 else base_demand * 0.70)
boom_demand = df[TARGET].max()
pandemic_ds_norm = 8.0 / 6.0
cascading_ds_norm = disrupt_hist.quantile(0.95)

tail_scenarios = {
    'S10': {'demand_level': 'Crisis', 'disruption_level': 'Pandemic',
            'demand_usd_mn': covid_trough, 'disruption_score': pandemic_ds_norm,
            'probability': 0.02, 'desc': 'Pandemic crisis (COVID-2020 trough)'},
    'S11': {'demand_level': 'Low', 'disruption_level': 'Cascading',
            'demand_usd_mn': cascading_demand, 'disruption_score': cascading_ds_norm,
            'probability': 0.03, 'desc': 'Cascading disruption (historical high-disruption)'},
    'S12': {'demand_level': 'Boom', 'disruption_level': 'None',
            'demand_usd_mn': boom_demand, 'disruption_score': 0.0,
            'probability': 0.02, 'desc': 'Demand boom (historical max export month)'},
}
scenarios.update(tail_scenarios)

total_prob = sum(s['probability'] for s in scenarios.values())
for s in scenarios.values():
    s['probability'] /= total_prob

for sk, sv in scenarios.items():
    d = sv['disruption_score']
    sv['capacity_utilization_cap'] = empirical_util_cap(d)
    sv['raw_material_cost_factor'] = empirical_cost_factor(d)
    sv['circular_material_availability'] = empirical_circ_availability(d)

print(f"✓ {len(scenarios)} scenarios generated\n")
print(f"{'Scenario':<8} {'Demand':>10} {'DS_norm':>8} {'Util%':>7} {'CostF':>7} {'CircAv':>7} "
      f"{'Prob':>7}  Description")
print("─"*95)
for sk, sv in scenarios.items():
    print(f"{sk:<8} {sv['demand_usd_mn']:>10.1f} {sv['disruption_score']:>8.3f} "
          f"{sv['capacity_utilization_cap']:>7.2f} {sv['raw_material_cost_factor']:>7.3f} "
          f"{sv['circular_material_availability']:>7.3f} {sv['probability']:>7.4f}  {sv['desc']}")

✓ 12 scenarios generated

Scenario     Demand  DS_norm   Util%   CostF  CircAv    Prob  Description
───────────────────────────────────────────────────────────────────────────────────────────────
S01          3690.6    0.000    1.00   1.000   0.200  0.1168  Low demand / None disruption
S02          3690.6    1.000    0.88   1.040   0.170  0.0818  Low demand / Moderate disruption
S03          3690.6    2.333    0.72   1.093   0.130  0.0350  Low demand / Severe disruption
S04          3950.0    0.000    1.00   1.000   0.200  0.2336  Medium demand / None disruption
S05          3950.0    1.000    0.88   1.040   0.170  0.1636  Medium demand / Moderate disruption
S06          3950.0    2.333    0.72   1.093   0.130  0.0701  Medium demand / Severe disruption
S07          4308.3    0.000    1.00   1.000   0.200  0.1168  High demand / None disruption
S08          4308.3    1.000    0.88   1.040   0.170  0.0818  High demand / Moderate disruption
S09          4308.3    2.333    0.72   1.093   0.

### Historical Series Underlying the Scenarios

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Historical Data Informing Scenario Construction', fontsize=14, fontweight='bold', y=1.02)

ax = axes[0, 0]
ax.plot(df['DateParsed'], df[TARGET], color='#2c3e50', linewidth=1)
ax.axhline(demand_scenarios['Low'], color='#e74c3c', ls='--', lw=1, label=f"Low={demand_scenarios['Low']:.0f}")
ax.axhline(demand_scenarios['Medium'], color='#f39c12', ls='--', lw=1, label=f"Med={demand_scenarios['Medium']:.0f}")
ax.axhline(demand_scenarios['High'], color='#27ae60', ls='--', lw=1, label=f"High={demand_scenarios['High']:.0f}")
ax.set_ylabel('RMG Export (USD mn)'); ax.set_title('(a) Export & Demand Anchors'); ax.legend(fontsize=8)

ax = axes[0, 1]
ax.plot(df['DateParsed'], df['Disruption_NormScore'], color='#8e44ad', linewidth=1)
ax.axhline(disruption_scenarios['Moderate'], color='#f39c12', ls='--', lw=1, label=f"Mod={disruption_scenarios['Moderate']:.3f}")
ax.axhline(disruption_scenarios['Severe'], color='#e74c3c', ls='--', lw=1, label=f"Sev={disruption_scenarios['Severe']:.3f}")
ax.set_ylabel('Normalized Disruption Score'); ax.set_title('(b) Disruption Anchors'); ax.legend(fontsize=8)

ax = axes[1, 0]
ax.plot(df['DateParsed'], df['Cotton_Price_USDkg'], color='#e67e22', linewidth=1, label='Cotton')
ax.set_ylabel('Cotton (USD/kg)'); ax.set_title('(c) Cotton Price (Cost Calibration)'); ax.legend(fontsize=8)

ax = axes[1, 1]
ax.plot(df['DateParsed'], df['RawMaterial_CostIdx'], color='#c0392b', linewidth=1)
ax.axhline(1.0, color='#95a5a6', ls=':', lw=1, label='Baseline'); ax.set_ylabel('Cost Index')
ax.set_title('(d) Composite Cost Index'); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('Fig_Historical_Scenario_Basis.png', bbox_inches='tight')
plt.show()
print("✓ Figure saved")

✓ Figure saved


## Two-Stage Stochastic MILP



In [7]:
def build_stochastic_milp(
    scenarios, strategies, dematel_factors,
    budget_usd_mn=150.0, planning_months=12, base_capacity_usd_mn=3500.0,
    cvar_alpha=0.95, lambda_risk=0.10, lambda_circular=0.25,
    lambda_resilience=0.25, lambda_profit=0.40, circular_target_pct=0.20,
    min_strategies=5, min_ce_strategies=2, min_res_strategies=2,
    circ_slack_penalty=0.10, verbose=True):
    """Two-stage stochastic MILP for circular & resilient capacity planning (v2)."""

    model = pyo.ConcreteModel(name="MILP_v2")

    # —— SETS ——
    model.J = pyo.Set(initialize=list(strategies.keys()))
    model.S = pyo.Set(initialize=list(scenarios.keys()))
    model.T = pyo.Set(initialize=range(1, planning_months + 1))

    ce_strategies  = [j for j, v in strategies.items() if v['type'] in ('circular', 'both')]
    res_strategies = [j for j, v in strategies.items() if v['type'] in ('resilience', 'both')]
    model.J_CE  = pyo.Set(initialize=ce_strategies)
    model.J_RES = pyo.Set(initialize=res_strategies)

    # —— PARAMETERS ——
    model.prob = pyo.Param(model.S, initialize={s: v['probability'] for s, v in scenarios.items()})
    model.demand = pyo.Param(model.S, model.T,
        initialize={(s, t): scenarios[s]['demand_usd_mn']
                    for s in scenarios for t in range(1, planning_months + 1)})
    model.disruption = pyo.Param(model.S, initialize={s: v['disruption_score'] for s, v in scenarios.items()})
    model.util_cap = pyo.Param(model.S, initialize={s: v['capacity_utilization_cap'] for s, v in scenarios.items()})
    model.circ_avail = pyo.Param(model.S, initialize={s: v['circular_material_availability'] for s, v in scenarios.items()})
    model.cost_factor = pyo.Param(model.S, initialize={s: v['raw_material_cost_factor'] for s, v in scenarios.items()})
    model.w_dematel = pyo.Param(model.J, initialize={j: v['dematel_weight'] for j, v in strategies.items()})

    # Capacity expansion (delta), recycling capacity (gamma), resilience effectiveness (res_eff)
    cap_rates = {'S1':0.08,'S2':0.10,'S3':0.08,'S4':0.18,'S5':0.22,'S6':0.20,'S7':0.12,'S8':0.06,'S9':0.05,'S10':0.15}
    model.delta = pyo.Param(model.J, initialize=cap_rates)

    rec_rates = {'S1':0.25,'S2':0.35,'S3':0.30,'S4':0.02,'S5':0.02,'S6':0.03,'S7':0.10,'S8':0.05,'S9':0.08,'S10':0.04}
    model.gamma = pyo.Param(model.J, initialize=rec_rates)

    # v2: Resilience effectiveness modulated by DEMATEL causal score
    base_res = {'S1':0.02,'S2':0.04,'S3':0.03,'S4':0.15,'S5':0.18,'S6':0.16,'S7':0.12,'S8':0.05,'S9':0.04,'S10':0.08}
    res_eff = {j: base_res[j] * (1.0 + 0.3 * strategies[j]['causal_score']) for j in strategies}
    model.res_eff = pyo.Param(model.J, initialize=res_eff)

    model.Budget = pyo.Param(initialize=budget_usd_mn)
    model.BaseCap = pyo.Param(initialize=base_capacity_usd_mn)
    model.RecCapBase = pyo.Param(initialize=base_capacity_usd_mn * 0.05)
    model.I_min = pyo.Param(initialize=2.0)
    model.I_max = pyo.Param(initialize=50.0)
    model.unit_revenue = pyo.Param(initialize=1.0)
    model.unit_prod_cost = pyo.Param(initialize=0.65)
    model.unit_shortfall_penalty = pyo.Param(initialize=0.50)
    model.circ_target = pyo.Param(initialize=circular_target_pct)
    model.circ_slack_penalty = pyo.Param(initialize=circ_slack_penalty)

    max_monthly_demand = max(sv['demand_usd_mn'] for sv in scenarios.values())
    model.ProfitUB = pyo.Param(initialize=max_monthly_demand * planning_months * 0.35)
    model.ProfitLB = pyo.Param(initialize=-max_monthly_demand * planning_months * 0.50)

    # —— VARIABLES ——
    model.x = pyo.Var(model.J, domain=pyo.Binary)
    model.I = pyo.Var(model.J, domain=pyo.NonNegativeReals, bounds=(0, 50))
    model.q  = pyo.Var(model.S, model.T, domain=pyo.NonNegativeReals)   # production
    model.r  = pyo.Var(model.S, model.T, domain=pyo.NonNegativeReals)   # recycled material
    model.v  = pyo.Var(model.S, model.T, domain=pyo.NonNegativeReals)   # virgin material
    model.u  = pyo.Var(model.S, model.T, domain=pyo.NonNegativeReals)   # unmet demand
    model.xi = pyo.Var(model.S, model.T, domain=pyo.NonNegativeReals)   # circular slack
    model.eta = pyo.Var(domain=pyo.Reals)
    model.z = pyo.Var(model.S, domain=pyo.NonNegativeReals)
    model.profit_s = pyo.Var(model.S, domain=pyo.Reals)

    # —— CONSTRAINTS ——
    # C1: Budget
    model.C1_Budget = pyo.Constraint(rule=lambda m: sum(m.I[j] for j in m.J) <= m.Budget)

    # C2: Minimum strategies (v2: parameterized, 0 for baseline)
    if min_strategies > 0:
        model.C2_MinStrat = pyo.Constraint(rule=lambda m: sum(m.x[j] for j in m.J) >= min_strategies)

    # C3: Investment activation link
    model.C3a = pyo.Constraint(model.J, rule=lambda m, j: m.I[j] <= m.I_max * m.x[j])
    model.C3b = pyo.Constraint(model.J, rule=lambda m, j: m.I[j] >= m.I_min * m.x[j])

    # C4: Min circular strategies (v2: parameterized, 0 for baseline)
    if min_ce_strategies > 0:
        model.C4_CEMin = pyo.Constraint(rule=lambda m: sum(m.x[j] for j in m.J_CE) >= min_ce_strategies)

    # C5: Min resilience strategies (v2: parameterized, 0 for baseline)
    if min_res_strategies > 0:
        model.C5_ResMin = pyo.Constraint(rule=lambda m: sum(m.x[j] for j in m.J_RES) >= min_res_strategies)

    # C6: Demand balance
    def c6(m, s, t):
        return m.q[s, t] + m.u[s, t] == m.demand[s, t]
    model.C6 = pyo.Constraint(model.S, model.T, rule=c6)

    # C7: Capacity with resilience recovery (v2)
    def c7(m, s, t):
        base_cap = m.BaseCap * m.util_cap[s]
        expansion = sum(m.delta[j] * m.I[j] for j in m.J)
        d_s = m.disruption[s]
        resilience_recovery = sum(m.res_eff[j] * m.I[j] for j in m.J) * d_s
        return m.q[s, t] <= base_cap + expansion + resilience_recovery
    model.C7 = pyo.Constraint(model.S, model.T, rule=c7)

    # C8: Material balance (virgin + recycled >= production)
    def c8(m, s, t):
        return m.v[s, t] + m.r[s, t] >= m.q[s, t]
    model.C8 = pyo.Constraint(model.S, model.T, rule=c8)

    # C9: Recycling capacity (investment-dependent)
    def c9(m, s, t):
        rec_cap = m.RecCapBase + sum(m.gamma[j] * m.I[j] for j in m.J)
        return m.r[s, t] <= rec_cap
    model.C9 = pyo.Constraint(model.S, model.T, rule=c9)

    # C10: Circular availability ceiling (disruption-dependent)
    def c10(m, s, t):
        return m.r[s, t] <= m.circ_avail[s] * m.q[s, t]
    model.C10 = pyo.Constraint(model.S, model.T, rule=c10)

    # C11: Soft circular target (proper slack variable, v2 fix)
    def c11(m, s, t):
        return m.r[s, t] + m.xi[s, t] >= m.circ_target * m.q[s, t]
    model.C11 = pyo.Constraint(model.S, model.T, rule=c11)

    # Profit definition (v2: no revenue_factor, recycled-material cost saving)
    def profit_def(m, s):
        revenue = sum(m.unit_revenue * m.q[s, t] for t in m.T)
        prod_cost = sum(m.unit_prod_cost * m.cost_factor[s] * m.q[s, t] for t in m.T)
        shortfall_cost = sum(m.unit_shortfall_penalty * m.u[s, t] for t in m.T)
        investment_cost = sum(m.I[j] for j in m.J)
        recycled_saving = sum(0.10 * m.r[s, t] for t in m.T)
        slack_cost = sum(m.circ_slack_penalty * m.xi[s, t] for t in m.T)
        return m.profit_s[s] == revenue - prod_cost - shortfall_cost - investment_cost + recycled_saving - slack_cost
    model.ProfitDef = pyo.Constraint(model.S, rule=profit_def)

    # CVaR
    model.CVaR_excess = pyo.Constraint(model.S, rule=lambda m, s: m.z[s] >= -m.profit_s[s] - m.eta)

    # —— OBJECTIVE (normalized, v2) ——
    def obj_rule(m):
        expected_profit = sum(m.prob[s] * m.profit_s[s] for s in m.S)
        profit_norm = (expected_profit - m.ProfitLB) / (m.ProfitUB - m.ProfitLB)

        expected_recycled = sum(m.prob[s] * m.r[s, t] for s in m.S for t in m.T)
        circ_norm = expected_recycled / (max_monthly_demand * planning_months * 0.50 + 1)

        total_exp_demand = sum(m.prob[s] * m.demand[s, t] for s in m.S for t in m.T)
        total_exp_shortfall = sum(m.prob[s] * m.u[s, t] for s in m.S for t in m.T)
        service_norm = 1 - total_exp_shortfall / (total_exp_demand + 1)

        cvar = m.eta + (1.0 / (1.0 - cvar_alpha)) * sum(m.prob[s] * m.z[s] for s in m.S)
        cvar_norm = (cvar - m.ProfitLB) / (m.ProfitUB - m.ProfitLB)

        return (lambda_profit * profit_norm + lambda_circular * circ_norm
                + lambda_resilience * service_norm - lambda_risk * cvar_norm)
    model.OBJ = pyo.Objective(rule=obj_rule, sense=pyo.maximize)

    if verbose:
        n_bin = len(strategies)
        n_cont = len(strategies) + 5*len(scenarios)*planning_months + 1 + 2*len(scenarios)
        n_cons = (1 + (1 if min_strategies > 0 else 0) + 2*n_bin
                  + (1 if min_ce_strategies > 0 else 0) + (1 if min_res_strategies > 0 else 0)
                  + 6*len(scenarios)*planning_months + 2*len(scenarios))
        print(f"\n  MILP v2: {n_bin+n_cont} vars ({n_bin} binary), ~{n_cons} constraints, "
              f"{len(scenarios)} scenarios, budget={budget_usd_mn}")

    return model

print("✓ MILP v2 builder defined.")

✓ MILP v2 builder defined.


## Solver & Results Extraction


In [8]:
def solve_milp(model, time_limit=300):
    solver = pyo.SolverFactory('appsi_highs')
    solver.options['time_limit'] = time_limit
    solver.options['mip_rel_gap'] = 0.001
    solver.options['threads'] = 4
    t0 = time.time()
    try:
        results = solver.solve(model, tee=False)
        elapsed = time.time() - t0
        status = results.solver.termination_condition
    except RuntimeError as e:
        elapsed = time.time() - t0
        print(f"  Solver: infeasible/error ({elapsed:.1f}s) — {e}")
        return None, pyo.TerminationCondition.infeasible
    obj_val = pyo.value(model.OBJ) if status in (
        pyo.TerminationCondition.optimal, pyo.TerminationCondition.feasible) else None
    print(f"  Solver: {str(status)}, time={elapsed:.1f}s", end='')
    if obj_val is not None:
        print(f", obj={obj_val:.4f}", end='')
    if status == pyo.TerminationCondition.feasible:
        print(" ⚠ feasible only (not proven optimal)", end='')
    print()
    return results, status


def extract_results(model, strategies, scenarios):
    results = {}
    strat_rows = []
    for j in model.J:
        activated = int(round(pyo.value(model.x[j])))
        inv = pyo.value(model.I[j])
        strat_rows.append({
            'Strategy': j, 'Name': strategies[j]['name'], 'Type': strategies[j]['type'],
            'Activated': activated, 'Investment_USDmn': inv,
            'DEMATEL_Weight': strategies[j]['dematel_weight'],
            'Causal_Score': strategies[j]['causal_score'],
            'Is_Cause': strategies[j]['is_cause'],
        })
    results['strategies'] = pd.DataFrame(strat_rows)

    scen_rows = []
    for s in model.S:
        profit = pyo.value(model.profit_s[s])
        total_prod = sum(pyo.value(model.q[s, t]) for t in model.T)
        total_rec = sum(pyo.value(model.r[s, t]) for t in model.T)
        total_vir = sum(pyo.value(model.v[s, t]) for t in model.T)
        total_short = sum(pyo.value(model.u[s, t]) for t in model.T)
        total_slack = sum(pyo.value(model.xi[s, t]) for t in model.T)
        total_dem = sum(pyo.value(model.demand[s, t]) for t in model.T)
        circ_pct = total_rec / total_prod * 100 if total_prod > 0 else 0
        sl = (1 - total_short / total_dem) * 100 if total_dem > 0 else 0
        scen_rows.append({
            'Scenario': s, 'Description': scenarios[s]['desc'],
            'Probability': scenarios[s]['probability'], 'Profit_USDmn': profit,
            'Production_USDmn': total_prod, 'Recycled_USDmn': total_rec,
            'Virgin_USDmn': total_vir, 'Circular_Pct': circ_pct,
            'Shortfall_USDmn': total_short, 'ServiceLevel_Pct': sl,
            'CircSlack_USDmn': total_slack, 'Demand_USDmn': total_dem,
        })
    results['scenarios'] = pd.DataFrame(scen_rows)

    df_s = results['scenarios']
    results['expected_profit'] = (df_s['Profit_USDmn'] * df_s['Probability']).sum()
    results['expected_service_level'] = (df_s['ServiceLevel_Pct'] * df_s['Probability']).sum()
    exp_rec = (df_s['Recycled_USDmn'] * df_s['Probability']).sum()
    exp_prod = (df_s['Production_USDmn'] * df_s['Probability']).sum()
    results['expected_circular_pct'] = (exp_rec / exp_prod * 100) if exp_prod > 0 else 0
    results['expected_circular_pct_scenweighted'] = (df_s['Circular_Pct'] * df_s['Probability']).sum()
    results['total_investment'] = results['strategies']['Investment_USDmn'].sum()
    results['n_activated'] = results['strategies']['Activated'].sum()
    results['objective_value'] = pyo.value(model.OBJ)
    results['cvar'] = pyo.value(model.eta)
    worst_idx = df_s['Profit_USDmn'].idxmin()
    results['worst_case'] = df_s.loc[worst_idx].to_dict()
    results['capacity_expansion'] = sum(pyo.value(model.delta[j]) * pyo.value(model.I[j]) for j in model.J)
    results['capacity_expansion_pct'] = results['capacity_expansion'] / pyo.value(model.BaseCap) * 100
    rc_base = pyo.value(model.RecCapBase)
    rc_exp = sum(pyo.value(model.gamma[j]) * pyo.value(model.I[j]) for j in model.J)
    results['recycling_capacity'] = rc_base + rc_exp
    results['recycling_capacity_expansion'] = rc_exp
    return results

print("✓ Solver and extraction utilities defined.")

✓ Solver and extraction utilities defined.


## Solve Base Model + No-Investment Baseline


In [9]:
print("——— Baseline (Zero Investment) ———")
model_bl = build_stochastic_milp(scenarios=scenarios, strategies=STRATEGIES,
    dematel_factors=DEMATEL_FACTORS, budget_usd_mn=0.0,
    min_strategies=0, min_ce_strategies=0, min_res_strategies=0, verbose=False)
_, bl_status = solve_milp(model_bl)
results_baseline = extract_results(model_bl, STRATEGIES, scenarios)
print(f"  Baseline: E[Profit]={results_baseline['expected_profit']:.1f}, "
      f"E[SL]={results_baseline['expected_service_level']:.1f}%, "
      f"E[CE]={results_baseline['expected_circular_pct']:.1f}%")

print("\n——— Base Model (Budget=150) ———")
model_base = build_stochastic_milp(scenarios=scenarios, strategies=STRATEGIES,
    dematel_factors=DEMATEL_FACTORS, budget_usd_mn=150.0, min_strategies=5)
_, base_status = solve_milp(model_base)
results_base = extract_results(model_base, STRATEGIES, scenarios)

dp = results_base['expected_profit'] - results_baseline['expected_profit']
ds = results_base['expected_service_level'] - results_baseline['expected_service_level']
dc = results_base['expected_circular_pct'] - results_baseline['expected_circular_pct']
print(f"\n  Value of investment: ΔProfit={dp:+.1f}, ΔSL={ds:+.1f}pp, ΔCE={dc:+.1f}pp")
print(f"  Capacity expansion: {results_base['capacity_expansion']:.1f} ({results_base['capacity_expansion_pct']:.2f}%)")
print(f"  Recycling capacity: {results_base['recycling_capacity_expansion']:.1f} → {results_base['recycling_capacity']:.1f} total")

——— Baseline (Zero Investment) ———
  Solver: optimal, time=0.2s, obj=0.4642
  Baseline: E[Profit]=7754.9, E[SL]=81.8%, E[CE]=5.5%

——— Base Model (Budget=150) ———

  MILP v2: 765 vars (10 binary), ~912 constraints, 12 scenarios, budget=150.0
  Solver: optimal, time=0.1s, obj=0.4726

  Value of investment: ΔProfit=+261.7, ΔSL=+1.0pp, ΔCE=+0.6pp
  Capacity expansion: 25.7 (0.73%)
  Recycling capacity: 20.5 → 195.5 total


## Sensitivity Analysis


In [10]:
print("——— Budget Sensitivity ———")
budget_levels = [50, 75, 100, 125, 150, 200, 250, 300]
sensitivity_results = []
for budget in budget_levels:
    print(f"  Budget={budget}...", end=' ')
    m = build_stochastic_milp(scenarios=scenarios, strategies=STRATEGIES,
        dematel_factors=DEMATEL_FACTORS, budget_usd_mn=budget, min_strategies=5, verbose=False)
    _, st = solve_milp(m, time_limit=120)
    if st in (pyo.TerminationCondition.optimal, pyo.TerminationCondition.feasible):
        r = extract_results(m, STRATEGIES, scenarios); r['budget'] = budget
        sensitivity_results.append(r)
        print(f"  E[Profit]={r['expected_profit']:.1f}, E[SL]={r['expected_service_level']:.1f}%, "
              f"E[CE]={r['expected_circular_pct']:.1f}%")
    else:
        print(f"  ⚠ Skipped (infeasible at budget={budget})")

——— Budget Sensitivity ———
  Budget=50...   Solver: optimal, time=0.1s, obj=0.4671
  E[Profit]=7810.4, E[SL]=82.0%, E[CE]=6.0%
  Budget=75...   Solver: optimal, time=0.1s, obj=0.4686
  E[Profit]=7857.5, E[SL]=82.1%, E[CE]=6.1%
  Budget=100...   Solver: optimal, time=0.1s, obj=0.4700
  E[Profit]=7915.0, E[SL]=82.3%, E[CE]=6.1%
  Budget=125...   Solver: optimal, time=0.1s, obj=0.4713
  E[Profit]=7966.7, E[SL]=82.5%, E[CE]=6.1%
  Budget=150...   Solver: optimal, time=0.1s, obj=0.4726
  E[Profit]=8016.6, E[SL]=82.7%, E[CE]=6.1%
  Budget=200...   Solver: optimal, time=0.1s, obj=0.4750
  E[Profit]=8054.2, E[SL]=82.9%, E[CE]=6.5%
  Budget=250...   Solver: optimal, time=0.2s, obj=0.4773
  E[Profit]=8139.7, E[SL]=83.2%, E[CE]=6.6%
  Budget=300...   Solver: optimal, time=0.1s, obj=0.4794
  E[Profit]=8199.7, E[SL]=83.5%, E[CE]=6.7%


In [11]:
print("——— Risk Aversion Sensitivity ———")
lambda_risk_levels = [0.0, 0.05, 0.10, 0.20, 0.30, 0.50]
risk_results = []
for lr in lambda_risk_levels:
    remaining = 1.0 - lr
    lp, lc, lres = 0.40*remaining/0.90, 0.25*remaining/0.90, 0.25*remaining/0.90
    print(f"  λ_risk={lr:.2f}...", end=' ')
    m = build_stochastic_milp(scenarios=scenarios, strategies=STRATEGIES,
        dematel_factors=DEMATEL_FACTORS, budget_usd_mn=150.0, lambda_risk=lr,
        lambda_circular=lc, lambda_resilience=lres, lambda_profit=lp,
        min_strategies=5, verbose=False)
    _, st = solve_milp(m, time_limit=120)
    if st in (pyo.TerminationCondition.optimal, pyo.TerminationCondition.feasible):
        r = extract_results(m, STRATEGIES, scenarios); r['lambda_risk'] = lr
        risk_results.append(r)
        wc = r['worst_case']
        print(f"  E[Profit]={r['expected_profit']:.1f}, Worst={wc['Profit_USDmn']:.1f}")

——— Risk Aversion Sensitivity ———
  λ_risk=0.00...   Solver: optimal, time=0.1s, obj=0.5930
  E[Profit]=7954.5, Worst=-1755.9
  λ_risk=0.05...   Solver: optimal, time=0.1s, obj=0.5327
  E[Profit]=7957.1, Worst=-1748.8
  λ_risk=0.10...   Solver: optimal, time=0.2s, obj=0.4726
  E[Profit]=8016.6, Worst=-1594.7
  λ_risk=0.20...   Solver: optimal, time=0.1s, obj=0.3523
  E[Profit]=8021.7, Worst=-1578.3
  λ_risk=0.30...   Solver: optimal, time=0.1s, obj=0.2323
  E[Profit]=8054.7, Worst=-1460.1
  λ_risk=0.50...   Solver: optimal, time=0.1s, obj=-0.0074
  E[Profit]=8054.7, Worst=-1460.1


In [12]:
print("——— Circular Target Sensitivity ———")
circ_targets = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
circ_results = []
for ct in circ_targets:
    print(f"  Target={ct*100:.0f}%...", end=' ')
    m = build_stochastic_milp(scenarios=scenarios, strategies=STRATEGIES,
        dematel_factors=DEMATEL_FACTORS, budget_usd_mn=150.0, circular_target_pct=ct,
        min_strategies=5, verbose=False)
    _, st = solve_milp(m, time_limit=120)
    if st in (pyo.TerminationCondition.optimal, pyo.TerminationCondition.feasible):
        r = extract_results(m, STRATEGIES, scenarios); r['circ_target'] = ct
        circ_results.append(r)
        print(f"  E[Profit]={r['expected_profit']:.1f}, E[CE]={r['expected_circular_pct']:.1f}%")

——— Circular Target Sensitivity ———
  Target=5%...   Solver: optimal, time=0.1s, obj=0.4781
  E[Profit]=8542.8, E[CE]=6.1%
  Target=10%...   Solver: optimal, time=0.1s, obj=0.4766
  E[Profit]=8394.6, E[CE]=6.1%
  Target=15%...   Solver: optimal, time=0.1s, obj=0.4746
  E[Profit]=8206.0, E[CE]=6.1%
  Target=20%...   Solver: optimal, time=0.1s, obj=0.4726
  E[Profit]=8016.6, E[CE]=6.1%
  Target=25%...   Solver: optimal, time=0.1s, obj=0.4705
  E[Profit]=7827.0, E[CE]=6.1%
  Target=30%...   Solver: optimal, time=0.1s, obj=0.4685
  E[Profit]=7637.4, E[CE]=6.1%
  Target=40%...   Solver: optimal, time=0.1s, obj=0.4644
  E[Profit]=7258.2, E[CE]=6.1%


In [13]:
df_strat = results_base['strategies']
print(f"\n{'Strat':<6} {'Name':<46} {'Type':<12} {'Act':>4} {'Invest':>7} {'DEMATEL':>8} {'Causal':>7}")
print("─"*96)
for _, row in df_strat.iterrows():
    m = '●' if row['Activated'] else '○'
    print(f"{row['Strategy']:<6} {row['Name']:<46} {row['Type']:<12} "
          f"{m:>4} {row['Investment_USDmn']:>7.1f} {row['DEMATEL_Weight']:>8.3f} {row['Causal_Score']:>7.3f}")
print(f"\n  Total investment:       {results_base['total_investment']:.1f} USD mn")
print(f"  Activated:              {results_base['n_activated']} / {len(STRATEGIES)}")
print(f"  E[Profit]:              {results_base['expected_profit']:.1f} USD mn")
print(f"  E[Service Level]:       {results_base['expected_service_level']:.1f}%")
print(f"  E[Circular %] (ratio):  {results_base['expected_circular_pct']:.1f}%")
print(f"  Capacity expansion:     {results_base['capacity_expansion']:.1f} ({results_base['capacity_expansion_pct']:.2f}%)")
print(f"  Recycling capacity:     {results_base['recycling_capacity']:.1f} USD mn")
print(f"  Objective (normalized): {results_base['objective_value']:.4f}")


Strat  Name                                           Type          Act  Invest  DEMATEL  Causal
────────────────────────────────────────────────────────────────────────────────────────────────
S1     Eco-Design & Circular Product Development      circular        ○     0.0    0.485   0.271
S2     Reverse Logistics & Recovery Infrastructure    circular        ●    50.0    0.521   0.022
S3     Closed-Loop Material Sourcing                  circular        ●     2.0    0.558   0.161
S4     Flexible Sourcing & Supplier Diversification   resilience      ●     2.0    0.715   0.390
S5     Agile Production & Operational Flexibility     resilience      ●    50.0    0.662   0.292
S6     Strategic Inventory & Buffer Capacity          resilience      ●    46.0    0.471   0.264
S7     Technology & Digital Supply Chain Integration  both            ○     0.0    0.715   0.489
S8     Workforce Upskilling & Green Training Programs both            ○     0.0    0.720   0.662
S9     Green Finance & Compli

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax = axes[0]
df_act = df_strat[df_strat['Activated'] == 1].sort_values('Investment_USDmn', ascending=True)
colors = {'circular': '#27ae60', 'resilience': '#2980b9', 'both': '#8e44ad'}
ax.barh(range(len(df_act)), df_act['Investment_USDmn'],
        color=[colors[t] for t in df_act['Type']], height=0.6)
for i, (_, row) in enumerate(df_act.iterrows()):
    ax.text(row['Investment_USDmn']+0.3, i, f"w={row['DEMATEL_Weight']:.2f}", va='center', fontsize=7, color='#555')
ax.set_yticks(range(len(df_act)))
ax.set_yticklabels([f"{r['Strategy']}: {r['Name'][:35]}" for _, r in df_act.iterrows()], fontsize=8)
ax.set_xlabel('Investment (USD mn)'); ax.set_title('(a) MILP-Derived Investment Allocation')
ax.legend(handles=[mpatches.Patch(color=c, label=l.title()) for l, c in colors.items()], loc='lower right', fontsize=9)

ax = axes[1]
tt = df_strat.groupby('Type')['Investment_USDmn'].sum()
tt = tt[tt > 0]
ax.pie(tt.values, labels=[t.title() for t in tt.index], autopct='%1.1f%%',
       colors=[colors.get(t,'#95a5a6') for t in tt.index], startangle=90)
ax.set_title('(b) Distribution by Type')
plt.tight_layout(); plt.savefig('Fig_Investment_Allocation.png', bbox_inches='tight'); plt.show()

In [15]:
df_scen = results_base['scenarios']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Scenario Performance (v2)', fontsize=14, fontweight='bold', y=1.02)

ax = axes[0, 0]
ax.bar(df_scen['Scenario'], df_scen['Profit_USDmn'],
       color=['#e74c3c' if p<0 else '#27ae60' for p in df_scen['Profit_USDmn']], alpha=0.8)
ax.axhline(results_base['expected_profit'], color='#2c3e50', ls='--', lw=1.5,
           label=f"E[Profit]={results_base['expected_profit']:.0f}")
ax.set_ylabel('Profit (USD mn)'); ax.set_title('(a) Profit'); ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=45, labelsize=8)

ax = axes[0, 1]
ax.bar(df_scen['Scenario'], df_scen['ServiceLevel_Pct'], color='#3498db', alpha=0.8)
ax.axhline(95, color='#e74c3c', ls='--', lw=1, label='95% target')
ax.axhline(results_base['expected_service_level'], color='#2c3e50', ls='--', lw=1.5,
           label=f"E[SL]={results_base['expected_service_level']:.1f}%")
ax.set_ylabel('Service Level (%)'); ax.set_title('(b) Service Level'); ax.set_ylim(0, 105)
ax.legend(fontsize=9); ax.tick_params(axis='x', rotation=45, labelsize=8)

ax = axes[1, 0]
ax.bar(df_scen['Scenario'], df_scen['Circular_Pct'], color='#27ae60', alpha=0.8)
ax.axhline(20, color='#e74c3c', ls='--', lw=1, label='20% target')
ax.set_ylabel('Circular %'); ax.set_title('(c) Circular Economy'); ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=45, labelsize=8)

ax = axes[1, 1]
sc = ax.scatter(df_scen['ServiceLevel_Pct'], df_scen['Profit_USDmn'],
                s=df_scen['Probability']*3000, c=df_scen['Circular_Pct'],
                cmap='RdYlGn', alpha=0.8, edgecolors='white', linewidths=1.5)
for _, row in df_scen.iterrows():
    ax.annotate(row['Scenario'], (row['ServiceLevel_Pct'], row['Profit_USDmn']), fontsize=7, ha='center', va='bottom')
plt.colorbar(sc, ax=ax, label='Circular %')
ax.set_xlabel('Service Level (%)'); ax.set_ylabel('Profit (USD mn)')
ax.set_title('(d) Risk-Return-Circularity')
plt.tight_layout(); plt.savefig('Fig_Scenario_Performance.png', bbox_inches='tight'); plt.show()

In [16]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Sensitivity Analysis (v2)', fontsize=14, fontweight='bold', y=1.04)

ax = axes[0]
budgets = [r['budget'] for r in sensitivity_results]
ax.plot(budgets, [r['expected_profit'] for r in sensitivity_results], 'o-', color='#2c3e50', lw=2, label='E[Profit]')
ax2 = ax.twinx()
ax2.plot(budgets, [r['expected_service_level'] for r in sensitivity_results], 's--', color='#3498db', lw=1.5, label='E[SL]%')
ax2.plot(budgets, [r['expected_circular_pct'] for r in sensitivity_results], 'd--', color='#27ae60', lw=1.5, label='E[CE]%')
ax.set_xlabel('Budget (USD mn)'); ax.set_ylabel('E[Profit]'); ax2.set_ylabel('%')
ax.set_title('(a) Budget'); l1,lb1=ax.get_legend_handles_labels(); l2,lb2=ax2.get_legend_handles_labels()
ax.legend(l1+l2, lb1+lb2, loc='lower right', fontsize=8)

ax = axes[1]
lrs = [r['lambda_risk'] for r in risk_results]
ax.plot(lrs, [r['expected_profit'] for r in risk_results], 'o-', color='#2c3e50', lw=2, label='E[Profit]')
ax.plot(lrs, [r['worst_case']['Profit_USDmn'] for r in risk_results], 's--', color='#e74c3c', lw=1.5, label='Worst')
ax.set_xlabel('λ_risk'); ax.set_ylabel('Profit (USD mn)'); ax.set_title('(b) Risk Aversion'); ax.legend(fontsize=9)

ax = axes[2]
tgts = [r['circ_target']*100 for r in circ_results]
ax.plot(tgts, [r['expected_profit'] for r in circ_results], 'o-', color='#2c3e50', lw=2, label='E[Profit]')
ax3 = ax.twinx()
ax3.plot(tgts, [r['expected_circular_pct'] for r in circ_results], 'd--', color='#27ae60', lw=1.5, label='E[CE]%')
ax3.plot(tgts, [r['expected_service_level'] for r in circ_results], 's--', color='#3498db', lw=1.5, label='E[SL]%')
ax.set_xlabel('Circular Target (%)'); ax.set_ylabel('E[Profit]'); ax3.set_ylabel('%')
ax.set_title('(c) Circular Target'); l1,lb1=ax.get_legend_handles_labels(); l2,lb2=ax3.get_legend_handles_labels()
ax.legend(l1+l2, lb1+lb2, loc='lower left', fontsize=8)
plt.tight_layout(); plt.savefig('Fig_Sensitivity.png', bbox_inches='tight'); plt.show()

In [17]:

fig, ax = plt.subplots(figsize=(12, 6))
strat_keys = list(STRATEGIES.keys())
scen_keys = list(scenarios.keys())
perf_matrix = np.zeros((len(strat_keys), len(scen_keys)))

for i, j in enumerate(strat_keys):
    inv_j = pyo.value(model_base.I[j])
    delta_j = pyo.value(model_base.delta[j])
    res_eff_j = pyo.value(model_base.res_eff[j])
    gamma_j = pyo.value(model_base.gamma[j])
    for k, s in enumerate(scen_keys):
        d_s = scenarios[s]['disruption_score']
        # Capacity + resilience recovery + recycling capacity contribution
        perf_matrix[i, k] = delta_j*inv_j + res_eff_j*inv_j*d_s + gamma_j*inv_j

im = ax.imshow(perf_matrix, cmap='YlGnBu', aspect='auto')
ax.set_xticks(range(len(scen_keys))); ax.set_xticklabels(scen_keys, fontsize=8, rotation=45)
ax.set_yticks(range(len(strat_keys)))
ax.set_yticklabels([f"{k}: {v['name'][:30]}" for k, v in STRATEGIES.items()], fontsize=8)
ax.set_xlabel('Scenario'); ax.set_ylabel('Strategy')
ax.set_title('Strategy Contribution (Capacity + Resilience + Recycling) Across Scenarios')
plt.colorbar(im, ax=ax, label='Contribution (USD mn equiv)')
for i in range(perf_matrix.shape[0]):
    for k in range(perf_matrix.shape[1]):
        v = perf_matrix[i,k]
        if v > 0.1:
            ax.text(k, i, f"{v:.1f}", ha='center', va='center', fontsize=5,
                    color='white' if v > perf_matrix.max()*0.6 else 'black')
plt.tight_layout(); plt.savefig('Fig_Heatmap.png', bbox_inches='tight'); plt.show()

In [18]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Value of MILP Investment: Baseline vs Optimized', fontsize=14, fontweight='bold', y=1.04)
metrics = ['expected_profit', 'expected_service_level', 'expected_circular_pct']
titles = ['Expected Profit (USD mn)', 'Expected Service Level (%)', 'Expected Circular Share (%)']
for ax, metric, title in zip(axes, metrics, titles):
    vals = [results_baseline[metric], results_base[metric]]
    bars = ax.bar(['Baseline', 'MILP Optimized'], vals, color=['#95a5a6', '#2980b9'], width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f"{val:.1f}", ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(title)
plt.tight_layout(); plt.savefig('Fig_Baseline_vs_Optimized.png', bbox_inches='tight'); plt.show()

In [19]:
output_path = 'MILP_Results_v2.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    results_base['strategies'].to_excel(writer, sheet_name='Strategy_Allocation', index=False)
    results_base['scenarios'].to_excel(writer, sheet_name='Scenario_Performance', index=False)

    pd.DataFrame([{
        'Model': 'Baseline', 'E_Profit': results_baseline['expected_profit'],
        'E_SL': results_baseline['expected_service_level'],
        'E_CE': results_baseline['expected_circular_pct'], 'Investment': 0,
    }, {
        'Model': 'MILP Optimized', 'E_Profit': results_base['expected_profit'],
        'E_SL': results_base['expected_service_level'],
        'E_CE': results_base['expected_circular_pct'],
        'Investment': results_base['total_investment'],
    }]).to_excel(writer, sheet_name='Baseline_Comparison', index=False)

    pd.DataFrame([{
        'Budget': r['budget'], 'Obj': r['objective_value'], 'E_Profit': r['expected_profit'],
        'E_SL': r['expected_service_level'], 'E_CE': r['expected_circular_pct'],
        'N_Active': r['n_activated'], 'Cap_Exp_Pct': r['capacity_expansion_pct'],
        'RecCap': r['recycling_capacity'],
    } for r in sensitivity_results]).to_excel(writer, sheet_name='Budget_Sensitivity', index=False)

    pd.DataFrame([{
        'Lambda_Risk': r['lambda_risk'], 'E_Profit': r['expected_profit'],
        'Worst': r['worst_case']['Profit_USDmn'], 'E_SL': r['expected_service_level'],
        'E_CE': r['expected_circular_pct'], 'N_Active': r['n_activated'],
    } for r in risk_results]).to_excel(writer, sheet_name='Risk_Sensitivity', index=False)

    pd.DataFrame([{
        'Target_Pct': r['circ_target']*100, 'E_Profit': r['expected_profit'],
        'E_SL': r['expected_service_level'], 'E_CE': r['expected_circular_pct'],
    } for r in circ_results]).to_excel(writer, sheet_name='CircTarget_Sensitivity', index=False)

    pd.DataFrame([{
        'Scenario': k, 'Demand': v['demand_level'], 'Disruption': v['disruption_level'],
        'Demand_USDmn': v['demand_usd_mn'], 'DS_Norm': v['disruption_score'],
        'Prob': v['probability'], 'Util': v['capacity_utilization_cap'],
        'CostF': v['raw_material_cost_factor'], 'CircAv': v['circular_material_availability'],
    } for k, v in scenarios.items()]).to_excel(writer, sheet_name='Scenario_Definitions', index=False)

    DEMATEL_FACTORS.to_excel(writer, sheet_name='DEMATEL_Factors', index=False)

print(f"✓ Results exported to: {output_path}")

✓ Results exported to: MILP_Results_v2.xlsx


In [20]:
print(f'''
═══ MILP v2 Summary ═══
Scenarios:    {len(scenarios)} (9 base + 3 tail)
Strategies:   {len(STRATEGIES)} ({results_base['n_activated']} activated)
Investment:   {results_base['total_investment']:.1f} USD mn
E[Profit]:    {results_base['expected_profit']:.1f} USD mn
E[SL]:        {results_base['expected_service_level']:.1f}%
E[CE%]:       {results_base['expected_circular_pct']:.1f}% (ratio)
CVaR(95%):    {results_base['cvar']:.1f} USD mn
Cap expansion:{results_base['capacity_expansion']:.1f} ({results_base['capacity_expansion_pct']:.2f}%)
RecCap:       {results_base['recycling_capacity']:.1f} USD mn

vs Baseline:
  ΔProfit:    {results_base['expected_profit']-results_baseline['expected_profit']:+.1f} USD mn
  ΔSL:        {results_base['expected_service_level']-results_baseline['expected_service_level']:+.1f} pp
  ΔCE:        {results_base['expected_circular_pct']-results_baseline['expected_circular_pct']:+.1f} pp
''')
print("✓ Notebook complete (v2).")


═══ MILP v2 Summary ═══
Scenarios:    12 (9 base + 3 tail)
Strategies:   10 (5 activated)
Investment:   150.0 USD mn
E[Profit]:    8016.6 USD mn
E[SL]:        82.7%
E[CE%]:       6.1% (ratio)
CVaR(95%):    -555.2 USD mn
Cap expansion:25.7 (0.73%)
RecCap:       195.5 USD mn

vs Baseline:
  ΔProfit:    +261.7 USD mn
  ΔSL:        +1.0 pp
  ΔCE:        +0.6 pp

✓ Notebook complete (v2).


In [21]:
import shutil

# Zip the entire Kaggle working/output directory
shutil.make_archive(
    "/kaggle/working/all_outputs",
    "zip",
    "/kaggle/working"
)

print("ZIP created: /kaggle/working/all_outputs.zip")

ZIP created: /kaggle/working/all_outputs.zip
